# 02 - Clean Baseline (Part 1: measure on clean images)

Establishes the baseline every later notebook compares against. Detection
needs an extra step first: a COCO-pretrained YOLOv8 can't be evaluated
against KITTI's own classes (no Pedestrian/Cyclist/Van/Tram in COCO), so its
head is fine-tuned on KITTI's classes here - that fine-tuned checkpoint, not
raw COCO-YOLO, is the detection "baseline" for every notebook downstream.
SegFormer needs no such step: a Cityscapes-pretrained checkpoint already
shares KITTI Semantics' class taxonomy.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import numpy as np
import pandas as pd
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow, kitti_yolo_format
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.tasks import feature_matching, optical_flow, object_detection, semantic_segmentation
from ipproj.viz.plotting import plot_bar_per_class, save_figure

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

## Step 1: fine-tune YOLOv8's head on KITTI's own classes (clean split)

In [ ]:
checkpoint_path = object_detection.checkpoint_path_for("clean_baseline")
if not checkpoint_path.exists():
    data_yaml = kitti_yolo_format.build_yolo_dataset(detection_splits)
    checkpoint_path = object_detection.fine_tune(data_yaml, run_name="clean_baseline")
(config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").write_text(str(checkpoint_path))

yolo_model = YOLO(checkpoint_path)

## Step 2: verify SegFormer/KITTI class alignment, then load

In [ ]:
segformer_model, segformer_processor = semantic_segmentation.load_pretrained()
aligned = semantic_segmentation.verify_class_alignment(segformer_model)
print("SegFormer/KITTI Semantics class alignment:", aligned)
assert aligned, (
    "Checkpoint class order does not match KITTI Semantics - populate "
    "config.SEGFORMER_CLASS_ID_REMAP before trusting segmentation metrics."
)

## Step 3: run all 4 tasks on clean images

In [ ]:
# Feature matching: no GT correspondence dataset to compare to on clean-only
# images, so the clean-image baseline is keypoint yield, not match accuracy
# (match accuracy is measured against distorted images starting in 03).
orb_keypoint_counts = []
for sample in detection_splits["test"][:20]:
    image = read_image(sample.image_path)
    keypoints, _ = feature_matching.detect_and_describe(image)
    orb_keypoint_counts.append(len(keypoints))

mean_orb_keypoints = float(np.mean(orb_keypoint_counts))
print(f"Mean ORB keypoints on clean test images: {mean_orb_keypoints:.1f}")

In [ ]:
epe_values = []
fl_error_values = []
for sample in flow_splits["test"]:
    frame1 = read_image(sample.frame1_path)
    frame2 = read_image(sample.frame2_path)
    gt_flow, valid = read_kitti_flow_png(sample.flow_gt_path)
    flow_metrics = optical_flow.evaluate(frame1, frame2, gt_flow, valid)
    epe_values.append(flow_metrics["epe"])
    fl_error_values.append(flow_metrics["fl_error"])

baseline_epe = float(np.mean(epe_values))
baseline_fl_error = float(np.mean(fl_error_values))
print(f"Baseline (clean) optical flow EPE: {baseline_epe:.3f}, Fl-error: {baseline_fl_error:.3f}")

In [ ]:
baseline_detection_metrics = object_detection.evaluate(yolo_model, detection_splits["test"])
baseline_map = float(baseline_detection_metrics["map"])
print(f"Baseline (clean) detection mAP: {baseline_map:.3f}")

detection_class_names = [config.KITTI_DETECTION_CLASSES[i] for i in baseline_detection_metrics["classes"].tolist()]
fig = plot_bar_per_class(
    detection_class_names, baseline_detection_metrics["map_per_class"].tolist(),
    ylabel="mAP", title="Baseline (clean) detection mAP per class",
)
save_figure(fig, "02_clean_baseline/detection_map_per_class.png")

In [ ]:
baseline_segmentation_iou = semantic_segmentation.evaluate(segformer_model, segformer_processor, segmentation_splits["test"])
baseline_mean_iou = float(baseline_segmentation_iou.mean())
print(f"Baseline (clean) segmentation mean IoU: {baseline_mean_iou:.3f}")

fig = plot_bar_per_class(
    config.CITYSCAPES_TRAINID_LABELS, baseline_segmentation_iou.tolist(),
    ylabel="IoU", title="Baseline (clean) segmentation IoU per class",
)
save_figure(fig, "02_clean_baseline/segmentation_iou_per_class.png")

## Baseline summary table

In [ ]:
config.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

baseline_summary = pd.DataFrame([
    {"task": "feature_matching", "metric": "mean_orb_keypoints_clean", "value": mean_orb_keypoints},
    {"task": "optical_flow", "metric": "epe", "value": baseline_epe},
    {"task": "optical_flow", "metric": "fl_error", "value": baseline_fl_error},
    {"task": "object_detection", "metric": "map", "value": baseline_map},
    {"task": "semantic_segmentation", "metric": "mean_iou", "value": baseline_mean_iou},
])
baseline_summary.to_csv(config.RESULTS_ROOT / "02_clean_baseline.csv", index=False)
baseline_summary